# ENGE707 Phase 2 – Improved eLCS for Medical-Insurance Risk Bands (merged notebook)

**Target:** `risk_band` = 3 equal-frequency bands (Low / Medium / High) of the continuous `risk_score`.
**LCS variant:** eLCS (`scikit-eLCS`, `skeLCS`).

| Section | Phase 2 task | Where it is covered |
|---|---|---|
| 1 | Task 1 – Phase-I summary | Markdown placeholder + dataset facts printed from the data |
| 2 | Task 2 – Original eLCS on raw data | Minimal processing, parameters, baseline CV score |
| 3 | Task 3 – Preprocessing / feature engineering | Placeholder (done by teammate) – this notebook loads its output |
| 4 | Task 4 – Improved LCS system | What changed, justification, pseudocode, ablation A→E, rule compaction |
| 5 | Task 5 – Experiments | 5×3 repeated stratified CV + 80/20 hold-out, 7 metrics, statistical tests |
| 6 | Task 6 – Model comparison | Raw eLCS vs preprocessed eLCS vs improved eLCS vs 5 conventional models |
| 7 | Task 7 – Interpretation / XAI | Decoded rules, ≥3 worked examples, rule-level trust check |
| 8 | Task 8 – Discussion | Skeleton to complete after the final run |

**Run order:** *Run All* from top to bottom. All outputs were cleared when the notebooks were merged, so re-run before taking numbers into the report.

## 1. Task 1 – Project Phase-I Summary

> **TODO (Task 1 owner):** restate the problem, dataset source, number of records/features, data-quality issues found in Phase I, and why an LCS suits this problem.
> The cell after the setup code prints the dataset facts (records, features, class balance) so the numbers in the report match the notebook.

In [ ]:
# ==============================================================================
# SETUP: imports, constants
# ==============================================================================
import time, warnings, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt          # pip install matplotlib
from scipy import stats
from joblib import Parallel, delayed
from skeLCS import eLCS                  # pip install scikit-eLCS

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, label_binarize
from sklearn.feature_selection import SelectFromModel
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report)

warnings.filterwarnings("ignore", category=UserWarning)

DATA_FILE  = "medical_insurance_cleaned_v2.csv"   # output of Task 3 (preprocessed data)
RAW_FILE   = "medical_insurance.csv"              # original raw data (Task 2 baseline)
SEED       = 42
N_SPLITS   = 5      # each training fold = 80% of the data, matching the 80/20 hold-out
N_REPEATS  = 3      # 5 x 3 = 15 paired scores per model
N_FEATURES = 4      # features kept by the selector in the improved system
N_JOBS     = -1     # folds run in parallel; set to 1 if you hit memory/pickling problems
N_BOOT     = 500    # bootstrap resamples for hold-out confidence intervals

In [ ]:
# ==============================================================================
# DATA (preprocessed, Task 3 output): target bands, leakage removal, encoding
# ==============================================================================
# Columns derived from (or directly determined by) the target -> removed to avoid data leakage
LEAKAGE_COLS = ["is_high_risk", "annual_premium", "monthly_premium", "annual_medical_cost",
                "claims_count", "avg_claim_amount", "total_claims_paid"]

df = pd.read_csv(DATA_FILE, keep_default_na=False, na_values=[""])   # keeps "None" in alcohol_freq as a category
y = pd.qcut(df["risk_score"], 3, labels=False).values                # 0 = Low, 1 = Medium, 2 = High
BAND_NAMES = {0: "Low", 1: "Medium", 2: "High"}

X_df = df.drop(columns=LEAKAGE_COLS + ["risk_score", "person_id"])
cat_maps = {}                                                        # integer code -> original label (used to read rules in Section 7)
for c in X_df.select_dtypes(exclude="number"):
    cats = X_df[c].astype("category")
    cat_maps[c] = dict(enumerate(cats.cat.categories))
    X_df[c] = cats.cat.codes                                         # text -> integer codes (eLCS needs numbers)
feature_names = list(X_df.columns)
X = X_df.values.astype(float)

print(f"Records: {len(df):,} | modelling features: {X.shape[1]} | categorical (integer-coded): {list(cat_maps)}")
print("Class shares (Low/Medium/High):", np.round(np.bincount(y) / len(y), 3))

In [ ]:
# ==============================================================================
# SHARED TOOLS: scikit-learn wrapper for eLCS, feature selector, metrics, CV engine
# ==============================================================================
class ScikitELCSWrapper(BaseEstimator, ClassifierMixin):
    """Makes skeLCS.eLCS usable inside sklearn Pipelines / clone(). No change to the eLCS algorithm."""
    def __init__(self, N=1000, learning_iterations=10000, do_correct_set_subsumption=False, random_state=None):
        self.N = N
        self.learning_iterations = learning_iterations
        self.do_correct_set_subsumption = do_correct_set_subsumption
        self.random_state = random_state

    def fit(self, X, y):
        y_int = np.asarray(y).astype(int)
        self.model_ = eLCS(N=self.N, learning_iterations=self.learning_iterations,
                           do_correct_set_subsumption=self.do_correct_set_subsumption,
                           random_state=self.random_state)
        self.model_.fit(X, y_int)
        self.classes_ = np.unique(y_int)
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        # eLCS class-vote proportions
        return self.model_.predict_proba(X)

def selector():
    """Keep the N_FEATURES most important columns of a shallow tree.
    Inside a Pipeline it is refit on each TRAINING fold only, so the test fold never influences selection."""
    return SelectFromModel(DecisionTreeClassifier(max_depth=5, random_state=SEED),
                           max_features=N_FEATURES, threshold=-np.inf)

def safe_proba(m, Xq, n_classes=3):
    try:
        p = np.nan_to_num(np.asarray(m.predict_proba(Xq), dtype=float))
        s = p.sum(axis=1, keepdims=True)
        p = np.where(s > 0, p / np.where(s == 0, 1, s), 1 / n_classes)   # rows with no votes -> uniform
        return p if p.shape[1] == n_classes else None
    except Exception:
        return None

def all_metrics(yt, pred, proba):
    out = {"accuracy":  accuracy_score(yt, pred),
           "bal_acc":   balanced_accuracy_score(yt, pred),
           "precision": precision_score(yt, pred, average="macro", zero_division=0),
           "recall":    recall_score(yt, pred, average="macro", zero_division=0),
           "f1":        f1_score(yt, pred, average="macro"),
           "roc_auc":   np.nan, "pr_auc": np.nan}
    if proba is not None:
        out["roc_auc"] = roc_auc_score(yt, proba, multi_class="ovr", average="macro")
        out["pr_auc"]  = average_precision_score(label_binarize(yt, classes=[0, 1, 2]), proba, average="macro")
    return out

def run_fold(model, Xm, ym, tr, te):
    t0 = time.time()
    m = clone(model).fit(Xm[tr], ym[tr])
    pred = m.predict(Xm[te])
    res = all_metrics(ym[te], pred, safe_proba(m, Xm[te]))
    res["time"] = time.time() - t0
    res["cm"] = confusion_matrix(ym[te], pred, labels=[0, 1, 2])
    return res

def run_cv(models, Xm, ym, splits):
    out = {}
    for name, model in models.items():
        t0 = time.time()
        out[name] = Parallel(n_jobs=N_JOBS)(delayed(run_fold)(model, Xm, ym, tr, te) for tr, te in splits)
        print(f"{name}: done in {time.time() - t0:.0f}s")
    return out

def to_frames(results):
    return {n: pd.DataFrame([{k: v for k, v in r.items() if k != "cm"} for r in rs]) for n, rs in results.items()}

# Identical CV splits for every model trained on the preprocessed data -> paired comparisons
cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
splits = list(cv.split(X, y))

## 2. Task 2 – Original eLCS on the raw dataset

**What is run:** the unmodified `skeLCS.eLCS` algorithm, via the thin sklearn wrapper above (the wrapper only converts labels to `int` and exposes `predict_proba`; it does not touch the algorithm).

**Minimal processing needed for compatibility (and nothing more):**
1. Drop the target (`risk_score`), the ID (`person_id`) and the leakage columns – otherwise the model would be given the answer.
2. Integer-code text columns (eLCS accepts numbers only).
3. Median-impute remaining numeric NaNs *inside the CV pipeline* (eLCS cannot handle NaN). Missing categories become their own code (`-1`).
4. No de-duplication, outlier handling, scaling, feature selection or class balancing – these belong to Task 3/4.

**Parameters:** library defaults for everything except `N=1000` and `learning_iterations=10000` (our fixed reference budget, which the improved system later increases). The next cell prints the library defaults so the report can state them exactly.

> Assumption: the raw file uses the same column names as the cleaned file. If it does not, adjust the `drop` list in the next cell.

In [ ]:
# ==============================================================================
# TASK 2: RAW / MINIMALLY PROCESSED DATA  ->  X_raw, y_raw
# ==============================================================================
raw = pd.read_csv(RAW_FILE, keep_default_na=False, na_values=[""])
raw = raw.dropna(subset=["risk_score"]).reset_index(drop=True)          # a row without a target cannot be scored
y_raw = pd.qcut(raw["risk_score"], 3, labels=False).values

drop_cols = [c for c in LEAKAGE_COLS + ["risk_score", "person_id"] if c in raw.columns]
Xr = raw.drop(columns=drop_cols)
for c in Xr.select_dtypes(exclude="number"):
    Xr[c] = Xr[c].astype("category").cat.codes                           # NaN -> -1
X_raw = Xr.values.astype(float)

print(f"Raw records: {len(raw):,} | raw features: {X_raw.shape[1]} | NaN cells left to impute: {int(np.isnan(X_raw).sum()):,}")
print(f"Duplicate rows kept (not removed at this stage): {int(raw.duplicated().sum()):,}")

raw_model = Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("elcs", ScikitELCSWrapper(random_state=SEED))])

# Reported parameters of the ORIGINAL eLCS
sig = inspect.signature(eLCS.__init__)
lib_defaults = pd.Series({k: v.default for k, v in sig.parameters.items() if k != "self"}, name="library default")
print("\nOriginal eLCS parameters = library defaults below, except N=1000, learning_iterations=10000, random_state=42")
display(lib_defaults.to_frame())

In [ ]:
# Baseline: original eLCS on the raw data, same repeated stratified CV protocol (own splits: the raw file has its own rows)
cv_raw = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
splits_raw = list(cv_raw.split(X_raw, y_raw))
RAW = "Original eLCS – raw data (minimal processing)"
raw_results = run_cv({RAW: raw_model}, X_raw, y_raw, splits_raw)
raw_frame = to_frames(raw_results)[RAW]
print(f"\n{RAW}: balanced accuracy {raw_frame['bal_acc'].mean():.3f} +/- {raw_frame['bal_acc'].std():.3f} | "
      f"macro-F1 {raw_frame['f1'].mean():.3f} | accuracy {raw_frame['accuracy'].mean():.3f}")

## 3. Task 3 – Data preprocessing and feature engineering

> **TODO (Task 3 owner):** describe how missing values, duplicates, invalid values, inconsistent categories, data types, outliers, encoding, scaling, class imbalance and feature engineering were handled, why this should help an LCS, and how leakage was avoided. Provide the cleaned dataset (`medical_insurance_cleaned_v2.csv`) or reproduction instructions.
>
> **Leakage notes that belong in that section** (these are handled in *this* notebook, but the cleaning step should be consistent with them):
> * Any statistic learned during cleaning (imputation values, outlier caps, scaling parameters) should come from training data only, or be justified as row-wise/rule-based.
> * `risk_band` cut-offs come from the quantiles of the full `risk_score` column; this defines the *target* and is not a feature-leakage path.
> * Leakage columns removed from the features: `is_high_risk`, `annual_premium`, `monthly_premium`, `annual_medical_cost`, `claims_count`, `avg_claim_amount`, `total_claims_paid`.
> * Feature selection and scaling used by the improved eLCS are refit inside every training fold (Section 4).

## 4. Task 4 – Improved LCS-based system

### 4.1 What was changed compared with the original eLCS
All changes are to the **pipeline and configuration**; the eLCS algorithm code is not modified. They are added one at a time (A → E) so each one's contribution is measurable.

| Config | Change vs previous | Why it may help |
|---|---|---|
| **A** | Original eLCS, preprocessed data, all 45 features | Reference ("original LCS on preprocessed data") |
| **B** | + feature selection: 4 features chosen by a depth-5 decision tree, refit on each training fold | With 45 features almost all rules are over-general or over-specific; the rule space is huge for a population of 1,000. 4 features shrink the search space so the GA can find accurate rules. |
| **C** | + `MinMaxScaler` (fit on training fold) | Puts interval-based predicates on a common 0–1 scale. This mainly aids readability/reproducibility; **check the ablation table – if C = B, scaling had no measurable effect and should be reported as such.** |
| **D** | + correct-set subsumption | A more general, equally accurate rule absorbs its specific copies, which reduces redundancy and makes rules more general. |
| **E** | + 20,000 learning iterations, population size N = 1,500 | More passes over ~77k training rows and room for more distinct rules; the tuned setting is the final **Improved eLCS**. |

A post-hoc **rule compaction** step (Section 6.1, it needs the hold-out fit) is then applied to the fitted model.

### 4.2 Algorithm of the improved system
```
INPUT  training data D_train (45 features), band labels y_train
1  Fit a depth-5 decision tree on D_train; keep the 4 features with highest importance  -> F
2  Fit MinMaxScaler on D_train[F]; transform D_train[F]
3  Run eLCS (N=1500, 20 000 iterations, correct-set subsumption ON, GA subsumption default):
      repeat for each iteration:
         pick a training instance; build match set [M]
         if no rule in [M] predicts the true band -> create a covering rule
         build correct set [C]; update accuracy / fitness of rules in [M]
         run GA in [C] (tournament selection, crossover, mutation); subsume where possible
         if population micro-size > N -> delete rules (deletion pressure on low fitness)
4  Compaction: keep rules with accuracy >= a* and numerosity >= n*  (a*, n* chosen on a validation split of D_train)
PREDICT  transform with the stored selector + scaler; fitness-weighted vote of matching rules
```
Steps 1–2 run inside a scikit-learn `Pipeline`, so they are re-fitted on each training fold and **never see test data**.

In [ ]:
# ==============================================================================
# MODEL DEFINITIONS  (ablation A -> E, and conventional comparison models)
# ==============================================================================
A = "A. Original eLCS (preprocessed, 45 feat)"
B = "B. + feature selection (4 feat)"
C = "C. + MinMax scaling"
D = "D. + correct-set subsumption"
E = "E. Improved eLCS (+20k iters, N=1500)"
BASE, REF = A, E        # reference = original eLCS on preprocessed data; improved system = E

ablation = {
    A: Pipeline([("elcs", ScikitELCSWrapper(random_state=SEED))]),
    B: Pipeline([("select", selector()),
                 ("elcs", ScikitELCSWrapper(random_state=SEED))]),
    C: Pipeline([("select", selector()), ("scale", MinMaxScaler()),
                 ("elcs", ScikitELCSWrapper(random_state=SEED))]),
    D: Pipeline([("select", selector()), ("scale", MinMaxScaler()),
                 ("elcs", ScikitELCSWrapper(random_state=SEED, do_correct_set_subsumption=True))]),
    E: Pipeline([("select", selector()), ("scale", MinMaxScaler()),
                 ("elcs", ScikitELCSWrapper(random_state=SEED, do_correct_set_subsumption=True,
                                            learning_iterations=20000, N=1500))]),
}

# Conventional (non-deep-learning) models. The "same 4 feat" variants separate the effect of
# feature selection from the effect of the learning algorithm.
conventional = {
    "Decision tree (45 feat)":       DecisionTreeClassifier(max_depth=5, random_state=SEED),
    "Random forest (45 feat)":       RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=1),
    "Logistic regression (45 feat)": Pipeline([("scale", StandardScaler()),
                                               ("lr", LogisticRegression(max_iter=1000))]),
    "Decision tree (same 4 feat)":   Pipeline([("select", selector()),
                                               ("dt", DecisionTreeClassifier(max_depth=5, random_state=SEED))]),
    "Random forest (same 4 feat)":   Pipeline([("select", selector()),
                                               ("rf", RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=1))]),
}
models = {**ablation, **conventional}
print(len(models), "models defined")

## 5. Task 5 – Experiments

### 5.1 Validation strategy
* **Primary: 5-fold stratified cross-validation repeated 3 times (15 paired scores per model).** The dataset has ~97k rows, but eLCS is stochastic and its results vary between runs, so a single split can be lucky or unlucky. Repetition gives a mean ± standard deviation and enough paired scores for significance tests. 5 folds means each model trains on **80 %** of the data, the same size as the hold-out training set below.
* **Stratification** keeps the Low/Medium/High proportions (~35/33/32 %) identical in every fold.
* **Same splits for every model** → paired differences, which is what the tests below need.
* **Secondary: one 80/20 stratified hold-out** (fit on 80 %, score on untouched 20 %). Used for the confusion matrices, per-class reports, bootstrap confidence intervals and McNemar tests, which need a single set of predictions.
* **No leakage:** feature selection and scaling are inside the pipelines, so they are refit on training folds only.

### 5.2 Metrics (classification, 3 balanced-ish classes)
Accuracy, balanced accuracy, macro precision, macro recall, macro F1, ROC-AUC (one-vs-rest, macro), PR-AUC (macro) and the confusion matrix.
**Balanced accuracy and macro-F1 are the headline metrics**: the bands are roughly balanced by construction, but macro-averaging stops a model from hiding a weak class (the Medium band is the hard one). ROC-AUC/PR-AUC judge the *ranking* quality of the class-vote probabilities independently of the decision threshold.

### 5.3 Statistical tests
* **Nadeau–Bengio corrected paired t-test** – repeated CV scores are not independent (training sets overlap), so the plain t-test is over-confident; this correction inflates the variance accordingly.
* **Wilcoxon signed-rank test** – non-parametric check on the same paired scores.
* **Holm correction** across the multiple comparisons against the improved model.
* **Friedman test** – is there any difference among all models?
* **McNemar exact test** on hold-out predictions – do two models make errors on *different* rows?
* The raw-data baseline uses different rows (the raw file was not cleaned), so it cannot be paired fold-by-fold; it is compared with an **unpaired Mann–Whitney U test**.

In [ ]:
# ==============================================================================
# EXPERIMENT 1: repeated stratified 5x3 CV on the preprocessed data (all models, all metrics)
# (takes several minutes: each eLCS fit is 30-60 s)
# ==============================================================================
fold_results = run_cv(models, X, y, splits)
per_fold = to_frames(fold_results)
per_fold[RAW] = raw_frame                      # raw-data baseline from Section 2 (own splits)

METRICS = ["accuracy", "bal_acc", "precision", "recall", "f1", "roc_auc", "pr_auc"]
COLS = ["Accuracy", "Balanced acc.", "Precision (macro)", "Recall (macro)", "F1 (macro)", "ROC-AUC (OvR)", "PR-AUC (macro)", "Time/fold (s)"]
summary = pd.DataFrame({
    n: {**{m: f"{d[m].mean():.3f} +/- {d[m].std():.3f}" for m in METRICS}, "time": f"{d['time'].mean():.0f}"}
    for n, d in per_fold.items()}).T
summary.columns = COLS
display(summary)

In [ ]:
# ==============================================================================
# TASK 4 EVIDENCE: ablation A -> E (each step vs the previous step and vs the original eLCS)
# ==============================================================================
abl_names = list(ablation)
abl = pd.DataFrame({
    "Balanced acc. (mean)": [per_fold[n]["bal_acc"].mean() for n in abl_names],
    "Std":                  [per_fold[n]["bal_acc"].std()  for n in abl_names],
    "Macro-F1 (mean)":      [per_fold[n]["f1"].mean()      for n in abl_names],
    "Gain vs A":            [per_fold[n]["bal_acc"].mean() - per_fold[A]["bal_acc"].mean() for n in abl_names],
    "Gain vs previous":     [np.nan] + [per_fold[abl_names[i]]["bal_acc"].mean() - per_fold[abl_names[i-1]]["bal_acc"].mean()
                                        for i in range(1, len(abl_names))],
    "Time/fold (s)":        [per_fold[n]["time"].mean() for n in abl_names],
}, index=abl_names).round(3)
display(abl)

In [ ]:
# ==============================================================================
# STATISTICAL TESTS (balanced accuracy, 15 paired scores)
# ==============================================================================
def nadeau_bengio(a, b, n_test_over_train):
    d = np.asarray(a) - np.asarray(b)
    k = len(d)
    var = d.var(ddof=1)
    if var == 0:
        return np.nan, np.nan, (np.nan, np.nan)
    se = np.sqrt((1 / k + n_test_over_train) * var)
    t = d.mean() / se
    p = 2 * stats.t.sf(abs(t), k - 1)
    half = stats.t.ppf(0.975, k - 1) * se
    return t, p, (d.mean() - half, d.mean() + half)

def holm(p):
    p = np.asarray(p, float); adj = np.full(len(p), np.nan)
    ok = ~np.isnan(p); idx = np.where(ok)[0]; order = idx[np.argsort(p[idx])]
    running = 0
    for rank, i in enumerate(order):
        running = max(running, (len(order) - rank) * p[i]); adj[i] = min(1.0, running)
    return adj

ratio = 1 / (N_SPLITS - 1)                      # n_test / n_train for k-fold
ref_scores = per_fold[REF]["bal_acc"].values

rows = []
for name in models:
    if name == REF:
        continue
    s = per_fold[name]["bal_acc"].values
    t, p, ci = nadeau_bengio(ref_scores, s, ratio)
    try:
        w = stats.wilcoxon(ref_scores, s).pvalue
    except ValueError:
        w = np.nan
    rows.append({"Compared with": name, "Mean diff (improved - other)": ref_scores.mean() - s.mean(),
                 "95% CI low": ci[0], "95% CI high": ci[1], "t": t, "NB p": p, "Wilcoxon p": w})
tests = pd.DataFrame(rows)
tests["NB p (Holm)"] = holm(tests["NB p"].values)
tests["Wilcoxon p (Holm)"] = holm(tests["Wilcoxon p"].values)
tests["Significant at 0.05 (NB, Holm)"] = tests["NB p (Holm)"] < 0.05
print("Improved eLCS vs every other model trained on the preprocessed data (paired, same folds):")
display(tests.round(4))

fr = stats.friedmanchisquare(*[per_fold[n]["bal_acc"].values for n in models])
print(f"Friedman test across all {len(models)} models (balanced accuracy): chi2 = {fr.statistic:.2f}, p = {fr.pvalue:.2g}")

# Raw-data baseline: different rows -> unpaired test
rows = []
for label, other in [("Improved eLCS (preprocessed)", REF), ("Original eLCS (preprocessed)", BASE)]:
    u = stats.mannwhitneyu(per_fold[other]["bal_acc"].values, per_fold[RAW]["bal_acc"].values, alternative="two-sided")
    rows.append({"Raw-data original eLCS vs": label,
                 "Mean diff (other - raw)": per_fold[other]["bal_acc"].mean() - per_fold[RAW]["bal_acc"].mean(),
                 "Mann-Whitney U": u.statistic, "p": u.pvalue})
print("\nRaw-data baseline (unpaired):")
display(pd.DataFrame(rows).round(4))

In [ ]:
# ==============================================================================
# EXPERIMENT 2: single stratified 80/20 hold-out (all models), bootstrap CIs, McNemar
# ==============================================================================
idx = np.arange(len(y))
tr, te = train_test_split(idx, test_size=0.2, stratify=y, random_state=SEED)

holdout = {}
for name, model in models.items():
    m = clone(model).fit(X[tr], y[tr])
    pred = m.predict(X[te])
    holdout[name] = {"model": m, "pred": pred, "proba": safe_proba(m, X[te])}
    print(f"{name}: hold-out balanced accuracy {balanced_accuracy_score(y[te], pred):.3f}")

# Raw-data baseline on its own 80/20 split
tr_r, te_r = train_test_split(np.arange(len(y_raw)), test_size=0.2, stratify=y_raw, random_state=SEED)
m_raw = clone(raw_model).fit(X_raw[tr_r], y_raw[tr_r])
pred_raw = m_raw.predict(X_raw[te_r])
holdout[RAW] = {"model": m_raw, "pred": pred_raw, "proba": safe_proba(m_raw, X_raw[te_r])}
print(f"{RAW}: hold-out balanced accuracy {balanced_accuracy_score(y_raw[te_r], pred_raw):.3f}")

y_true = {n: (y_raw[te_r] if n == RAW else y[te]) for n in holdout}

# Full metric table
ho_table = pd.DataFrame({n: all_metrics(y_true[n], h["pred"], h["proba"]) for n, h in holdout.items()}).T
ho_table.columns = COLS[:-1]
display(ho_table.round(3))

# 95% bootstrap CI for balanced accuracy (resampling test rows)
rng = np.random.default_rng(SEED)
ci_rows, ho_ci = [], {}
for name, h in holdout.items():
    yt = y_true[name]
    bi = rng.integers(0, len(yt), size=(N_BOOT, len(yt)))
    b = [balanced_accuracy_score(yt[i], h["pred"][i]) for i in bi]
    ho_ci[name] = (np.percentile(b, 2.5), np.percentile(b, 97.5))
    ci_rows.append({"Model": name, "Balanced acc.": balanced_accuracy_score(yt, h["pred"]),
                    "95% CI low": ho_ci[name][0], "95% CI high": ho_ci[name][1]})
display(pd.DataFrame(ci_rows).round(3))

# McNemar's exact test vs the improved eLCS (same test rows)
rows = []
ok_ref = holdout[REF]["pred"] == y[te]
for name in models:
    if name == REF:
        continue
    ok_other = holdout[name]["pred"] == y[te]
    b_ = int(np.sum(ok_ref & ~ok_other)); c_ = int(np.sum(~ok_ref & ok_other))
    p = stats.binomtest(b_, b_ + c_, 0.5).pvalue if (b_ + c_) > 0 else np.nan
    rows.append({"Compared with": name, "Improved right only": b_, "Other right only": c_, "McNemar p": p})
mc = pd.DataFrame(rows)
mc["McNemar p (Holm)"] = holm(mc["McNemar p"].values)
display(mc.round(4))

In [ ]:
# ==============================================================================
# CONFUSION MATRICES (hold-out) + per-class report for the improved eLCS
# ==============================================================================
best_conv = max(conventional, key=lambda n: balanced_accuracy_score(y[te], holdout[n]["pred"]))
panel = [(RAW, y_raw[te_r]), (BASE, y[te]), (REF, y[te]), (best_conv, y[te])]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, yt) in zip(axes, panel):
    cm = confusion_matrix(yt, holdout[name]["pred"], labels=[0, 1, 2])
    cmn = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(cmn, vmin=0, vmax=1, cmap="Blues")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]}\n({cmn[i, j]:.0%})", ha="center", va="center",
                    color="white" if cmn[i, j] > 0.5 else "black", fontsize=9)
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(list(BAND_NAMES.values())); ax.set_yticklabels(list(BAND_NAMES.values()))
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(name, fontsize=9)
plt.tight_layout(); plt.show()

print("Per-class report, improved eLCS (hold-out):")
print(classification_report(y[te], holdout[REF]["pred"], target_names=list(BAND_NAMES.values()), digits=3))

## 6. Task 6 – Model comparison

The table below contains exactly what Task 6 asks for: **(i)** original eLCS on raw data, **(ii)** original eLCS on preprocessed data, **(iii)** the improved eLCS, and **(iv)** five conventional non-deep-learning models (decision tree, random forest, logistic regression, plus decision tree / random forest restricted to the same 4 features the improved eLCS uses).
Read the "same 4 feat" rows together with the 45-feature rows: they show how much of any gap comes from the *features* and how much from the *algorithm*.

In [ ]:
# ==============================================================================
# TASK 6: FINAL COMPARISON TABLE (+ chart)
# ==============================================================================
def category(n):
    if n == RAW:  return "1. Original eLCS – raw data"
    if n == BASE: return "2. Original eLCS – preprocessed"
    if n == REF:  return "3. Improved eLCS"
    if n in conventional: return "4. Conventional ML"
    return "Ablation step"

order = [RAW, BASE, REF] + list(conventional)
comp = pd.DataFrame([{
    "Category": category(n), "Model": n,
    "CV balanced acc.": f"{per_fold[n]['bal_acc'].mean():.3f} +/- {per_fold[n]['bal_acc'].std():.3f}",
    "CV macro-F1": f"{per_fold[n]['f1'].mean():.3f}",
    "CV ROC-AUC": f"{per_fold[n]['roc_auc'].mean():.3f}",
    "Hold-out bal. acc.": f"{balanced_accuracy_score(y_true[n], holdout[n]['pred']):.3f}",
    "Hold-out 95% CI": f"[{ho_ci[n][0]:.3f}, {ho_ci[n][1]:.3f}]",
    "Features": 4 if (n == REF or "4 feat" in n) else X.shape[1],
} for n in order]).set_index("Category")
display(comp)

fig, ax = plt.subplots(figsize=(8, 4))
means = [per_fold[n]["bal_acc"].mean() for n in order]; sds = [per_fold[n]["bal_acc"].std() for n in order]
ax.barh(range(len(order)), means, xerr=sds, color=["#bbb", "#999", "#1f77b4"] + ["#ff7f0e"] * len(conventional))
ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=8); ax.invert_yaxis()
ax.set_xlabel("CV balanced accuracy (mean ± sd, 15 folds)"); ax.set_xlim(0.4, 1.02)
plt.tight_layout(); plt.show()

## 6.1 Rule compaction (completes Task 4 – rule pruning)
The fitted eLCS keeps many redundant or weak rules. Compaction keeps only rules with `accuracy >= a*` and `numerosity >= n*`.
**To avoid tuning on the test set, (a*, n*) are chosen on a validation split carved out of the training data**: the smallest rule set whose validation balanced accuracy is within 0.005 of the full rule set. The chosen thresholds are then applied once to the final model and scored on the untouched hold-out.

In [ ]:
# ==============================================================================
# TASK 4: RULE COMPACTION (thresholds chosen on a validation split of the TRAINING data)
# ==============================================================================
final = holdout[REF]["model"]                   # improved eLCS pipeline fitted on the 80% training split

def popset(pipe):  return pipe.named_steps["elcs"].model_.population.popSet
def prune(rules, min_acc, min_num):
    return [r for r in rules if r.accuracy >= min_acc and r.numerosity >= min_num]

def predict_with(pipe, rules, Xq):
    """Predict using a temporary rule set; the original population is always restored."""
    model_ = pipe.named_steps["elcs"].model_
    original = model_.population.popSet
    model_.population.popSet = rules
    try:
        return pipe.predict(Xq)
    finally:
        model_.population.popSet = original

Xfit, Xval, yfit, yval = train_test_split(X[tr], y[tr], test_size=0.2, stratify=y[tr], random_state=SEED)
probe = clone(ablation[REF]).fit(Xfit, yfit)                         # same configuration, trained on 64% of the data
full_val = balanced_accuracy_score(yval, probe.predict(Xval))

grid = [(0.0, 1), (0.5, 1), (0.8, 1), (0.8, 2), (0.9, 2), (0.9, 5)]
rows = []
for a_, n_ in grid:
    kept = prune(popset(probe), a_, n_)
    if len(kept) == 0:
        continue
    rows.append({"min_acc": a_, "min_num": n_, "rules (macro)": len(kept),
                 "val balanced acc": balanced_accuracy_score(yval, predict_with(probe, kept, Xval))})
comp_grid = pd.DataFrame(rows)
display(comp_grid.round(3))

ok = comp_grid[comp_grid["val balanced acc"] >= full_val - 0.005]
chosen = ok.sort_values("rules (macro)").iloc[0]
print(f"Validation balanced accuracy of the full rule set: {full_val:.3f}")
print(f"Chosen thresholds: min_acc={chosen['min_acc']}, min_num={int(chosen['min_num'])}")

full_pop = popset(final)
kept_final = prune(full_pop, chosen["min_acc"], int(chosen["min_num"]))
bal_full = balanced_accuracy_score(y[te], final.predict(X[te]))
bal_comp = balanced_accuracy_score(y[te], predict_with(final, kept_final, X[te]))
display(pd.DataFrame({
    "Rule set": ["Full (improved eLCS)", "Compacted"],
    "Rules (macro)": [len(full_pop), len(kept_final)],
    "Rules (micro, sum of numerosity)": [sum(r.numerosity for r in full_pop), sum(r.numerosity for r in kept_final)],
    "Hold-out balanced acc.": [bal_full, bal_comp]}).round(3))

## 7. Task 7 – Interpretation of results and Explainable AI

**What to say about the comparison (fill in with the final numbers):** which model family is best on balanced accuracy / macro-F1, whether the improved eLCS is significantly better than the original on both raw and preprocessed data (Section 5 tests), whether it is significantly different from the conventional models, and which band is hardest (see the per-class report and confusion matrices – look at the Medium band).

**Why eLCS is interpretable:** the final model *is* a set of human-readable `IF condition THEN band` rules. Each rule carries **accuracy** (how often it is right when it matches), **fitness**, and **numerosity** (how many copies/similar rules the GA has merged into it – a proxy for how much the system trusts it).

The cells below decode the rules into plain language (undoing the MinMax scaling and the integer coding of categories), then check each important rule against the **hold-out data it never saw**.

In [ ]:
# ==============================================================================
# TASK 7: DECODE RULES OF THE IMPROVED eLCS
# ==============================================================================
final = holdout[REF]["model"]
sel_mask = final.named_steps["select"].get_support()
selected = [f for f, keep in zip(feature_names, sel_mask) if keep]
scaler = final.named_steps["scale"]
elcs_model = final.named_steps["elcs"].model_
Xte_s = final[:-1].transform(X[te])             # hold-out rows after selection + scaling (what the rules see)
print("Features used by the improved eLCS:", selected)

sel_tree = final.named_steps["select"].estimator_
print("\nSelector tree importances (fitted on training data only):")
display(pd.Series(sel_tree.feature_importances_, index=feature_names).sort_values(ascending=False).head(N_FEATURES).round(3).to_frame("importance"))

def is_interval(cond):
    return isinstance(cond, (list, tuple, np.ndarray))

def describe(rule):
    parts = []
    for idx_, cond in zip(rule.specifiedAttList, rule.condition):
        name = selected[idx_]
        lo0, rng_ = scaler.data_min_[idx_], scaler.data_range_[idx_]
        dmin, dmax = lo0, lo0 + rng_
        if is_interval(cond):
            lo, hi = max(cond[0] * rng_ + lo0, dmin), min(cond[1] * rng_ + lo0, dmax)   # clip to the observed range
            parts.append(f"{name} in [{lo:.1f}, {hi:.1f}]")
        else:
            v = cond * rng_ + lo0
            if name in cat_maps:
                parts.append(f"{name} = {cat_maps[name].get(int(round(v)), round(v, 2))}")
            elif abs(v - round(v)) < 1e-6:
                parts.append(f"{name} = {int(round(v))}")
            else:
                parts.append(f"{name} = {v:.2f}")
    return " AND ".join(parts) if parts else "ANY patient (default rule)"

def rule_mask(rule, Xs):
    m = np.ones(len(Xs), dtype=bool)
    for idx_, cond in zip(rule.specifiedAttList, rule.condition):
        col = Xs[:, idx_]
        m &= (col >= cond[0]) & (col <= cond[1]) if is_interval(cond) else np.isclose(col, cond)
    return m

pop = list(elcs_model.population.popSet)
print(f"\nRule population: {len(pop)} rules (macro) | {sum(r.numerosity for r in pop)} (micro)")

def rule_row(r):
    m = rule_mask(r, Xte_s)
    cls = int(r.phenotype)
    return {"IF": describe(r), "THEN": BAND_NAMES[cls], "train accuracy": round(r.accuracy, 3),
            "fitness": round(r.fitness, 3), "numerosity": r.numerosity,
            "hold-out matches": int(m.sum()),
            "hold-out precision": round(float((y[te][m] == cls).mean()), 3) if m.sum() else np.nan}

top = sorted(pop, key=lambda r: r.numerosity * r.fitness, reverse=True)[:10]
pd.set_option("display.max_colwidth", 110)
print("\nTop 10 rules by numerosity x fitness:")
display(pd.DataFrame([rule_row(r) for r in top]))

In [ ]:
# ==============================================================================
# SAMPLE RULES FOR THE REPORT: strongest rule per risk band (>= 3 worked examples)
# ==============================================================================
samples = []
for cls in (0, 1, 2):
    cands = [r for r in pop if int(r.phenotype) == cls and len(r.specifiedAttList) > 0]
    if cands:
        samples.append(max(cands, key=lambda r: r.numerosity * r.fitness))

for r in samples:
    row = rule_row(r)
    print(f"IF {row['IF']}\n   THEN risk band = {row['THEN']}")
    print(f"   train accuracy {row['train accuracy']} | numerosity {row['numerosity']} | "
          f"hold-out: matches {row['hold-out matches']} patients, correct {row['hold-out precision']}\n")

# Which features do the rules actually use? (share of the micro-population specifying each feature)
tot = sum(r.numerosity for r in pop)
usage = pd.Series({f: sum(r.numerosity for r in pop if i in list(r.specifiedAttList)) / tot
                   for i, f in enumerate(selected)}).sort_values(ascending=False)
display(usage.round(3).to_frame("share of rules specifying feature"))

# Rule-level trust check: precision of all rules with decent numerosity on unseen hold-out data
chk = pd.DataFrame([rule_row(r) for r in pop if r.numerosity >= 3])
chk = chk[chk["hold-out matches"] >= 30]
print(f"\n{len(chk)} rules (numerosity >= 3, >= 30 hold-out matches): "
      f"median train accuracy {chk['train accuracy'].median():.3f} vs median hold-out precision {chk['hold-out precision'].median():.3f}")

**How to discuss the sample rules (Task 7 text):**
* *Decision process:* for a new patient, all rules whose conditions match vote for a band, weighted by fitness × numerosity. Say which feature combinations push towards Low vs High (e.g. number of chronic conditions, age range, smoking status, BMI range).
* *Compare with domain sense:* do the age/chronic-condition/smoker patterns agree with how insurers think about risk? Rules that contradict domain knowledge deserve a comment.
* *Trustworthiness:* compare *train accuracy* with *hold-out precision* in the tables above. A small gap means the explanation generalises; a large gap means the rule memorised noise. Note that rules are only trustworthy where they match many patients, and that the **Medium band is the weakest** (low recall in the per-class report), so medium-risk explanations are the least reliable.
* *Practical meaning:* the rules describe how the **risk_score formula/bands behave in this dataset**, which is an association and not a causal statement about patients.

## 8. Task 8 – Discussion, limitations and future work (skeleton – complete after the final run)

**Reasons for improvement / lack of it**
* Use the ablation table: which step produced the largest jump (feature selection? subsumption? more iterations?). Report honestly if MinMax scaling made no difference (B vs C).
* Compare the improved eLCS with the conventional models. If a decision tree / random forest still wins, say so and explain: e.g. the target is a banded version of a score that may be a near-deterministic function of a few features, which axis-aligned trees capture exactly, whereas eLCS must *discover* interval boundaries stochastically with a limited rule population. The "same 4 feat" rows show whether the gap comes from the algorithm or from the feature set.
* Training time: eLCS is slower than all conventional models; say whether the extra cost is worth the interpretability.

**Practical meaning:** what a balanced accuracy of the final value means for patients (e.g. share of Medium-band patients misclassified) – refer to the confusion matrix, not only the headline number.

**Limitations:** single dataset and single random seed family; the bands are quantile cut-offs of a continuous score (rows near a boundary are inherently ambiguous – this explains much of the Medium-band confusion); possible synthetic/simulated data; features removed as leakage mean the model uses only demographic/health attributes; hyper-parameters were tuned by hand, not by an exhaustive search; the raw-data baseline cannot be paired fold-by-fold with the cleaned-data models.

**Responsible use:** the model should support, not replace, an underwriter; explanations are associations; risk bands could encode unfair treatment of age or smoking status – review with a domain expert before any real use; external validation on independent data is needed.

**Future work:** other LCS variants (ExSTraCS, XCS, BioHEL) and attribute tracking / expert knowledge for feature selection; systematic hyper-parameter search (Bayesian / grid) on a validation set; cost-sensitive or ordinal-aware learning (the bands are ordered, Low→High); better compaction algorithms; external validation and domain-expert review of the rules.